In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from google.colab import drive
drive.mount("/content/drive/")

PROJECT_DIR = Path("/content/drive/MyDrive/Underwater-Image-Data-set-main")
V1_DIR = PROJECT_DIR / "Dataset_V1"

WEEK4_DIR = V1_DIR / "Week_4_Final_Review"
WEEK4_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_DIR = V1_DIR / "Classical_Baselines"
TUNING_DIR = V1_DIR / "Classical_Tuning"
SPECIAL_DIR = V1_DIR / "Special_Analysis"

print("=" * 70)
print("WEEK 4 FINAL REVIEW PACKAGE")
print("=" * 70)

def find_file(folder, filenames):
    for name in filenames:
        path = folder / name
        if path.exists():
            return path
    return None

baseline_file = find_file(
    BASELINE_DIR,
    [
        "baseline_comparison.csv",
        "all_baseline_results.csv"
    ]
)

tuning_file = TUNING_DIR / "parameter_tuning_results.csv"

special_summary_file = (
    SPECIAL_DIR /
    "classical_vs_learning_summary.csv"
)

special_results_file = (
    SPECIAL_DIR /
    "classical_vs_learning_test_results.csv"
)

ablation_file = (
    SPECIAL_DIR /
    "limited_parameter_ablation.csv"
)

best_config_file = (
    TUNING_DIR /
    "best_validated_method.json"
)

if best_config_file.exists():
    with open(best_config_file, "r") as f:
        best_config = json.load(f)
else:
    best_config = {}

selected_method = best_config.get(
    "selected_method",
    "Not available"
)

print("\nSelected tuned method:", selected_method)

if baseline_file is not None:

    baseline = pd.read_csv(
        baseline_file
    )

    print("\nBaseline file:", baseline_file.name)
    print("Baseline columns:")
    print(list(baseline.columns))

    metric_candidates = {
        "PSNR": [
            "PSNR",
            "psnr"
        ],
        "SSIM": [
            "SSIM",
            "ssim"
        ],
        "Edge Preservation": [
            "Edge Preservation",
            "edge_preservation",
            "Edge_Preservation"
        ]
    }

    baseline_rows = []

    method_column = next(
        (
            c for c in [
                "Method",
                "method",
                "METHOD"
            ]
            if c in baseline.columns
        ),
        None
    )

    if method_column is not None:

        for _, row in baseline.iterrows():

            result = {
                "Method":
                    row[method_column],
                "Type":
                    "Baseline"
            }

            for metric, candidates in metric_candidates.items():

                found = next(
                    (
                        c for c in candidates
                        if c in baseline.columns
                    ),
                    None
                )

                if found is not None:
                    result[metric] = row[found]

            baseline_rows.append(result)

    baseline_summary = pd.DataFrame(
        baseline_rows
    )

else:

    baseline_summary = pd.DataFrame()

    print("\nBaseline comparison file not found.")

if tuning_file.exists():

    tuning = pd.read_csv(
        tuning_file
    )

    tuned_best = tuning[
        tuning["selected"] == True
    ].copy()

    if len(tuned_best) == 0:
        tuned_best = (
            tuning
            .sort_values("Average_Rank")
            .head(1)
        )

    tuned_best["Type"] = "Tuned"

    tuned_summary = tuned_best[
        [
            "method",
            "Type",
            "PSNR",
            "SSIM",
            "Edge Preservation"
        ]
    ].copy()

    tuned_summary = tuned_summary.rename(
        columns={
            "method": "Method"
        }
    )

else:

    tuned_summary = pd.DataFrame()

    print("\nTuning results not found.")

if len(baseline_summary) > 0 or len(tuned_summary) > 0:

    baseline_vs_tuned = pd.concat(
        [
            baseline_summary,
            tuned_summary
        ],
        ignore_index=True
    )

    baseline_vs_tuned.to_csv(
        WEEK4_DIR /
        "baseline_vs_tuned.csv",
        index=False
    )

    print("\nBASELINE VS TUNED")
    print(baseline_vs_tuned)

else:

    baseline_vs_tuned = pd.DataFrame()

if special_summary_file.exists():

    special_summary = pd.read_csv(
        special_summary_file
    )

    special_summary.to_csv(
        WEEK4_DIR /
        "classical_vs_learning_summary.csv",
        index=False
    )

    print("\nCLASSICAL VS LEARNING")
    print(special_summary)

else:

    special_summary = pd.DataFrame()

    print(
        "\nClassical-vs-learning summary not found."
    )

effect_rows = []

if len(special_summary) > 0:

    classical = special_summary[
        special_summary["condition"]
        == "Best validated classical"
    ]

    learning = special_summary[
        special_summary["condition"]
        == "Learning-based"
    ]

    if len(classical) > 0 and len(learning) > 0:

        c = classical.iloc[0]
        u = learning.iloc[0]

        for metric in [
            "PSNR",
            "SSIM",
            "Edge Preservation",
            "Edge F1",
            "Processing Time"
        ]:

            if metric in special_summary.columns:

                effect_rows.append({
                    "Metric": metric,
                    "Classical":
                        c[metric],
                    "U-Net":
                        u[metric],
                    "Difference_U-Net_minus_Classical":
                        u[metric] - c[metric]
                })

effect_df = pd.DataFrame(
    effect_rows
)

if len(effect_df) > 0:

    effect_df.to_csv(
        WEEK4_DIR /
        "classical_vs_learning_effect.csv",
        index=False
    )

    print("\nQUANTIFIED EFFECT")
    print(effect_df)

if ablation_file.exists():

    ablation = pd.read_csv(
        ablation_file
    )

    ablation.to_csv(
        WEEK4_DIR /
        "limited_parameter_ablation.csv",
        index=False
    )

    print("\nLIMITED PARAMETER ABLATION")
    print(ablation)

else:

    ablation = pd.DataFrame()

    print(
        "\nAblation results not found."
    )

if special_results_file.exists():

    detailed = pd.read_csv(
        special_results_file
    )

    metric_columns = [
        c for c in [
            "PSNR",
            "SSIM",
            "Edge Preservation",
            "Edge F1"
        ]
        if c in detailed.columns
    ]

    if len(metric_columns) > 0:

        detailed["Quality_Score"] = (
            detailed[metric_columns]
            .rank(
                ascending=False,
                pct=True
            )
            .mean(axis=1)
        )

        success_cases = (
            detailed
            .sort_values(
                "Quality_Score",
                ascending=False
            )
            .head(10)
        )

        failure_cases = (
            detailed
            .sort_values(
                "Quality_Score",
                ascending=True
            )
            .head(10)
        )

        success_cases.to_csv(
            WEEK4_DIR /
            "representative_success_cases.csv",
            index=False
        )

        failure_cases.to_csv(
            WEEK4_DIR /
            "representative_failure_cases.csv",
            index=False
        )

        print("\nRepresentative success cases:")
        print(
            success_cases[
                [
                    "sample_id",
                    "method",
                    "PSNR",
                    "SSIM",
                    "Edge Preservation"
                ]
            ].head()
        )

        print("\nRepresentative failure cases:")
        print(
            failure_cases[
                [
                    "sample_id",
                    "method",
                    "PSNR",
                    "SSIM",
                    "Edge Preservation"
                ]
            ].head()
        )

else:

    detailed = pd.DataFrame()

    print(
        "\nDetailed test results not found."
    )

failure_summary = []

if len(detailed) > 0:

    for method in detailed["method"].unique():

        method_data = detailed[
            detailed["method"] == method
        ]

        row = {
            "Method": method,
            "Samples": len(method_data)
        }

        for metric in [
            "PSNR",
            "SSIM",
            "Edge Preservation",
            "Edge F1"
        ]:

            if metric in method_data.columns:

                row[
                    f"{metric}_Mean"
                ] = method_data[metric].mean()

                row[
                    f"{metric}_Min"
                ] = method_data[metric].min()

        failure_summary.append(row)

failure_summary_df = pd.DataFrame(
    failure_summary
)

if len(failure_summary_df) > 0:

    failure_summary_df.to_csv(
        WEEK4_DIR /
        "failure_pattern_summary.csv",
        index=False
    )

    print("\nFAILURE PATTERN SUMMARY")
    print(failure_summary_df)

review_summary = {
    "selected_tuned_method":
        selected_method,

    "baseline_vs_tuned_completed":
        len(baseline_vs_tuned) > 0,

    "classical_vs_learning_completed":
        len(special_summary) > 0,

    "parameter_ablation_completed":
        len(ablation) > 0,

    "success_failure_analysis_completed":
        len(detailed) > 0
}

with open(
    WEEK4_DIR /
    "Week_4_Review_Summary.json",
    "w"
) as f:

    json.dump(
        review_summary,
        f,
        indent=4
    )

print("WEEK 4 REVIEW PACKAGE COMPLETE")

print("\nFolder:")
print(WEEK4_DIR)

print("\nFiles created:")
for file in sorted(WEEK4_DIR.iterdir()):
    print(" -", file.name)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
WEEK 4 FINAL REVIEW PACKAGE

Selected tuned method: Gamma

Baseline file: baseline_comparison.csv
Baseline columns:
['method', 'PSNR', 'SSIM', 'Edge_Preservation']

BASELINE VS TUNED
      Method      Type       PSNR      SSIM  Edge Preservation
0      CLAHE  Baseline  13.888109  0.613580           0.025002
1      Gamma  Baseline  13.893641  0.654586           0.006676
2  GrayWorld  Baseline  13.327233  0.653937           0.008210
3      Gamma     Tuned  13.697721  0.662334           0.005826

CLASSICAL VS LEARNING
  method                 condition       PSNR      SSIM  Edge Preservation  \
0  Gamma  Best validated classical  13.844543  0.654768           0.007723   
1  U-Net            Learning-based  14.162797  0.630716           0.032845   

   Edge Precision  Edge Recall   Edge F1  Processing Time  
0        0.025546     0.007723  0.010610         0.00